# Peak Selection — Evolutionary Modules (All Cell Types)

Runs `build_regions` for every cell type with robust DESeq2 results and saves
the output in long-format to `peak_selection/regions_all.parquet`.

**Three downstream notebooks read this file:**
- `13b_snc_enrichment.ipynb` — SNC enrichment tests + plots
- `13c_heatmaps.ipynb` — evolutionary accessibility heatmaps

**Module pipeline** (see `src/evolutionary_heatmap.py → build_regions`):
1. Load ultra-robust peak IDs (pre-filtered in R: padj < 0.05, |LFC| > 1, strict
   cross-donor reproducibility)
2. `ortho_filter` — keep only peaks with orthologous sequence in all required species
3. Hierarchical subtraction — remove peaks significant at deeper phylogenetic nodes
   (controlled by `PADJ_THRESH_SIG` and `LFC_THRESH_SIG`)
4. `acc_waterfall` — min(up-species) must exceed every down-species
5. Rank by |LFC| × −log10(padj)
6. `posthoc_filter` — remove if any down-species > `POSTHOC_THRESHOLD` × up-species mean

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from pathlib import Path

from src.evolutionary_heatmap import (
    load_pseudobulk_matrices,
    build_regions,
    discover_cell_types,
    sanitize_name,
)

print("Imports OK")

Imports OK


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE       = "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
CONS_DIR   = f"{BASE}/cross_species_consensus_v3"
DESEQ_DIR  = f"{CONS_DIR}/13_deseq2_R_pseudobulk"
ANNO_PATH  = f"{CONS_DIR}/07_master_annotation/master_annotation_corrected.tsv"
FRAG_DIR   = f"{CONS_DIR}/12_fragment_matrices"
ROBUST_DIR = f"{DESEQ_DIR}/differential_results/ultra_robust_branches"
BRANCH_DIR = f"{DESEQ_DIR}/differential_results/evolutionary_branches"

SPECIES = ["Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset"]

# ── Peak selection thresholds ──────────────────────────────────────────────────
#
# ── STAGE 1: DESeq2 (applied in R, results loaded as CSV) ─────────────────────
# Pseudobulk design: one count column per donor × region combination.
# Only peaks with orthologs in ALL required species are tested.
# Active-peak filter: keep peaks with count ≥ 10 in at least 2 pseudobulks.
# Model: ~ 0 + species  (contrast = positive vs negative branch species)
# Significance threshold: padj < 0.05, log2FoldChange > 1
#
# ── STAGE 2: Ultra-Robust filter (applied in R, results loaded as CSV) ─────────
# Takes only the DESeq2-significant UP peaks from Stage 1 and applies a
# COMPLETE SEPARATION criterion at the pseudobulk level:
#
#   min_pos_logCPM > max_neg_logCPM     (every positive pseudobulk exceeds
#                                        every negative pseudobulk)
#   min_pos_logCPM > 1                  (peak must be genuinely accessible:
#                                        > 1 log2CPM in all positive pseudobulks)
#
# Where log2CPM is computed per-pseudobulk using full cell-type library sizes.
# This is NOT cross-validation — it is a single-pass min/max check across ALL
# pseudobulks of the positive vs negative species.
# Output columns: peak_id, min_pos_logCPM, max_neg_logCPM
#
# ── STAGE 3: Python-side filters (adjustable here) ─────────────────────────────
PADJ_THRESH_SIG   = 0.1   # padj for excluding peaks also significant at deeper nodes
LFC_THRESH_SIG    = 0.5   # |LFC| threshold for the same exclusion step
POSTHOC_THRESHOLD = 0.8   # max fraction of up-species mean allowed in a down-species
MAX_PER_BLOCK     = 50    # top-N peaks per module shown in heatmap (no effect on BED/enrichment)

# ── Output ─────────────────────────────────────────────────────────────────────
OUT_SELECTION = Path(DESEQ_DIR) / "peak_selection"
OUT_SELECTION.mkdir(parents=True, exist_ok=True)
print(f"Output: {OUT_SELECTION}")

In [4]:
print("Loading master annotation...")
anno = pd.read_csv(ANNO_PATH, sep="\t", low_memory=False).set_index("peak_id")
orth_cols = {sp: f"{sp}_orth" for sp in SPECIES}
det_cols  = {sp: f"{sp}_det"  for sp in SPECIES}
print(f"  {anno.shape[0]:,} peaks × {anno.shape[1]} columns")

CELL_TYPES = discover_cell_types(ROBUST_DIR, FRAG_DIR, SPECIES)
print(f"\nCell types with robust results ({len(CELL_TYPES)}):")
for ct in CELL_TYPES:
    print(f"  {ct}")

Loading master annotation...
  1,142,003 peaks × 59 columns

Cell types with robust results (13):
  Colonocytes
  Crypt Fibroblasts WNT2B+
  ECs
  Enterocytes
  Goblet cells
  Macrophages
  Naive B cells
  Plasma B cells
  Specialized Fibroblasts RSPO3+ only
  Specialized Fibroblasts SYNM+
  Stem cells
  T cells
  TA cells


In [5]:
# ── Peak selection: run build_regions for all cell types ──────────────────────

all_rows        = []
all_filter_stats = []

for cell_type in CELL_TYPES:
    print(f"\n{'='*65}")
    print(f"  {cell_type}")
    print(f"{'='*65}")

    # Species-level pseudobulk used for waterfall / posthoc accessibility filters
    acc_species_df, _, _ = load_pseudobulk_matrices(cell_type, SPECIES, FRAG_DIR)
    if acc_species_df.empty:
        print(f"  [SKIP] no pseudobulk data found")
        continue

    regions, regions_all, total_candidates, filter_stats = build_regions(
        cell_type=cell_type,
        anno=anno,
        filter_acc_df=acc_species_df,
        robust_dir=ROBUST_DIR,
        branch_dir=BRANCH_DIR,
        orth_cols=orth_cols,
        det_cols=det_cols,
        species=SPECIES,
        max_per_block=MAX_PER_BLOCK,
        padj_thresh_sig=PADJ_THRESH_SIG,
        lfc_thresh_sig=LFC_THRESH_SIG,
        posthoc_threshold=POSTHOC_THRESHOLD,
    )

    # Collect long-format rows (all peaks, not just top-N)
    for module, peak_ids in regions_all.items():
        for rank, pid in enumerate(peak_ids):
            all_rows.append({
                "cell_type": cell_type,
                "module": module,
                "peak_id": pid,
                "rank": rank,
            })

    # Collect filter stats
    for module, stats in filter_stats.items():
        all_filter_stats.append({
            "cell_type": cell_type,
            "module": module,
            **stats,
        })

# Save regions_all as long-format parquet
regions_df = pd.DataFrame(all_rows)
out_parquet = OUT_SELECTION / "regions_all.parquet"
regions_df.to_parquet(out_parquet, index=False)
print(f"\n{'='*65}")
print(f"Saved {len(regions_df):,} peak-module entries → {out_parquet}")


  Colonocytes
  Human: 6 Colonocytes pseudobulks (5494 cells)
  Bonobo: 1 Colonocytes pseudobulks (975 cells)
  Chimpanzee: 2 Colonocytes pseudobulks (463 cells)
  Gorilla: 1 Colonocytes pseudobulks (419 cells)
  Macaque: 2 Colonocytes pseudobulks (1620 cells)
  Marmoset: 2 Colonocytes pseudobulks (1407 cells)
  Human-specific DNA:            1594 -> 50 shown
  Human UP vs All:               2869 -> 50 shown
  ILS: Human + Gorilla UP:     285 -> 50 shown
  ILS: Human + Chimp UP:      84 -> 50 shown
  ILS: Human + Bonobo UP:     236 -> 50 shown
  Great Apes UP vs Macaque:      1288 -> 50 shown
  Old World UP vs Marmoset:      2989 -> 50 shown

  Category                            Candidates   Shown
  ----------------------------------------------------------
  Human-specific DNA                        1594      50
  Human UP vs All                           2869      50
  ILS: Human + Gorilla UP                    285      50
  ILS: Human + Chimp UP                       84      50
  

In [6]:
# ── Filter transparency table ─────────────────────────────────────────────────

stats_df = pd.DataFrame(all_filter_stats)

print("Final peak counts per module × cell type:\n")
pivot = stats_df.pivot_table(
    index="module", columns="cell_type",
    values="n_final", aggfunc="first",
).fillna(0).astype(int)
display(pivot)

print("\nFilter funnel (totals across all cell types):")
funnel = stats_df.groupby("module")[
    ["n_candidates", "n_after_ortho", "n_after_waterfall", "n_final"]
].sum()
funnel["pct_retained"] = (100 * funnel["n_final"] / funnel["n_candidates"].clip(lower=1)).round(1)
display(funnel)

Final peak counts per module × cell type:



cell_type,Colonocytes,Crypt Fibroblasts WNT2B+,ECs,Enterocytes,Goblet cells,Macrophages,Naive B cells,Plasma B cells,Specialized Fibroblasts RSPO3+ only,Specialized Fibroblasts SYNM+,Stem cells,T cells,TA cells
module,,,,,,,,,,,,,
Great Apes UP vs Macaque,1288,119,0,834,589,976,1533,577,50,10,421,465,803
Human UP vs All,2869,2949,181,5896,2966,1056,745,1566,1188,1377,2066,1171,2719
Human-specific DNA,1594,1594,1594,1594,1594,1594,1594,1594,1594,1594,1594,1594,1594
ILS: Human + Bonobo UP,236,68,14,270,260,73,269,110,84,191,350,56,104
ILS: Human + Chimp UP,84,147,7,281,145,73,101,106,158,247,57,41,43
ILS: Human + Gorilla UP,285,343,84,1026,436,174,372,399,140,442,303,180,259
Old World UP vs Marmoset,2989,1015,327,4672,3123,1262,1586,1642,193,223,3340,1210,2661



Filter funnel (totals across all cell types):


,n_candidates,n_after_ortho,n_after_waterfall,n_final,pct_retained
module,,,,,
Great Apes UP vs Macaque,22981,13273,8779,7665,33.4
Human UP vs All,38282,38282,33207,26749,69.9
Human-specific DNA,20722,20722,20722,20722,100.0
ILS: Human + Bonobo UP,5542,4909,2705,2085,37.6
ILS: Human + Chimp UP,5066,4293,2155,1490,29.4
ILS: Human + Gorilla UP,16938,12873,6549,4443,26.2
Old World UP vs Marmoset,24303,24303,24274,24243,99.8
